# ETL VRA/ANAC — 2021

**Responsável pelo ETL:** Arthur. **Responsável pelo ELT:** Murilo Acioli.

Fluxo: **12 CSVs originais → tratamento em Python/Pandas → evidências locais →
carga opcional da camada tratada no PostgreSQL**. O banco recebe os dados depois
das transformações; nenhuma regra de limpeza depende de SQL.

Referências: Contrato Único v1.1 do grupo e regras/saídas do ELT 2021 de Murilo.
O notebook ETL de 2019 foi usado como referência de organização, sem importar seu
script antigo. Este notebook contém a implementação de 2021 em células explicadas.

**Como executar:** reinicie o kernel da `.venv` e execute de cima para baixo.
`EXPORTAR_EVIDENCIAS=True` gera os resultados em `dados/processados/2021/etl/`.
`EXECUTAR_CARGA_POSTGRES=False` permite revisar tudo sem escrever no banco.
Os CSVs de entrada são apenas lidos e seus hashes são conferidos.

**Limites:** esta entrega produz a camada tratada e auditorias de 2021. Dimensões
e Fato compartilhadas entre os três anos pertencem à integração do grupo.
A equivalência completa com o ELT depende dos hashes e da comparação por registro;
os números informados por Murilo são referências de conferência, não filtros.

No Colab, prepare uma cópia do repositório com os 12 CSVs em `dados/brutos/` e
as dependências de `requirements.txt`. Para executar a carga, use um PostgreSQL
acessível pelo ambiente do Colab; o `localhost` do Colab não é o seu computador.

## 1. Conferir o ambiente

O interpretador exibido deve pertencer à `.venv` do projeto. Os módulos `csv` e
`pathlib` fazem parte do Python; o Pandas será usado para manipular as tabelas.

In [1]:
import csv
import hashlib
import json
import re
import sys
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def sha256_arquivo(caminho):
    digest = hashlib.sha256()
    with caminho.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            digest.update(bloco)
    return digest.hexdigest()


print("Python utilizado:", sys.executable)
print("Versão do Pandas:", pd.__version__)

Python utilizado: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/.venv/bin/python
Versão do Pandas: 2.3.3


## 2. Localizar os 12 arquivos e definir os campos do contrato

A raiz é localizada a partir da pasta atual ou de suas pastas superiores. Isso
permite executar tanto da raiz do repositório quanto da pasta do notebook, sem
fixar o caminho pessoal do computador. Os arquivos devem estar em `dados/brutos/`.

Os 20 nomes abaixo vêm do contrato. `Codeshare` é uma coluna adicional conhecida
de 2021, preservada para auditoria; os metadados de origem não fazem parte dos
campos usados para identificar repetições.

In [2]:
ano_origem = 2021

raiz_projeto = Path.cwd().resolve()
while not (raiz_projeto / "requirements.txt").is_file():
    if raiz_projeto == raiz_projeto.parent:
        raise FileNotFoundError(
            "Abra o notebook a partir da pasta do projeto ou de uma subpasta."
        )
    raiz_projeto = raiz_projeto.parent

pasta_dados = raiz_projeto / "dados" / "brutos"
arquivos = [
    pasta_dados / f"VRA_{ano_origem}_{mes:02d}.csv"
    for mes in range(1, 13)
]

faltantes = [arquivo.name for arquivo in arquivos if not arquivo.is_file()]
if faltantes:
    raise FileNotFoundError(f"Arquivos não encontrados: {faltantes}")

colunas_comuns = [
    "Sigla ICAO Empresa Aérea",
    "Empresa Aérea",
    "Número Voo",
    "Código DI",
    "Código Tipo Linha",
    "Modelo Equipamento",
    "Número de Assentos",
    "Sigla ICAO Aeroporto Origem",
    "Descrição Aeroporto Origem",
    "Partida Prevista",
    "Partida Real",
    "Sigla ICAO Aeroporto Destino",
    "Descrição Aeroporto Destino",
    "Chegada Prevista",
    "Chegada Real",
    "Situação Voo",
    "Justificativa",
    "Referência",
    "Situação Partida",
    "Situação Chegada",
]

print("Pasta dos dados:", pasta_dados)
print(f"Todos os {len(arquivos)} arquivos foram encontrados.")

# A carga fica desativada para revisão; nenhum banco é criado automaticamente.
PASTA_SAIDA = raiz_projeto / "dados" / "processados" / "2021" / "etl"
EXPORTAR_EVIDENCIAS = True
EXPORTAR_BASE_BRUTA = False  # Os CSVs originais já ficam preservados na entrada.
EXECUTAR_CARGA_POSTGRES = False
SUBSTITUIR_CARGA_POSTGRES = False
SCHEMA_ETL = "etl_2021"
validacao_concluida = False
inicio_execucao = datetime.now(timezone.utc).isoformat()

nomes_esperados = {arquivo.name for arquivo in arquivos}
nomes_encontrados = {p.name for p in pasta_dados.glob("VRA_2021_*.csv")}
if nomes_encontrados != nomes_esperados:
    raise ValueError("A pasta contém nomes de CSVs de 2021 fora dos 12 meses esperados.")

hashes_entrada = {arquivo.name: sha256_arquivo(arquivo) for arquivo in arquivos}
print("Pasta de saída:", PASTA_SAIDA)
print("Carga PostgreSQL habilitada:", EXECUTAR_CARGA_POSTGRES)


Pasta dos dados: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/dados/brutos
Todos os 12 arquivos foram encontrados.
Pasta de saída: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/dados/processados/2021/etl
Carga PostgreSQL habilitada: False


## 3. Inspecionar uma amostra de janeiro

Lemos os campos como texto para preservar zeros à esquerda, vazios e os formatos
originais. `nrows=5` limita apenas esta inspeção; a base completa será lida depois.
O formato de `Referência` observado na amostra é `DD/MM/AAAA HH:MM:SS`, diferente
do formato de 2019 descrito no contrato. Os demais meses ainda precisam de
verificação dos formatos dos valores antes da conversão.

In [3]:
amostra = pd.read_csv(
    arquivos[0],
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    skip_blank_lines=False,
    nrows=5,
)

print(f"Colunas encontradas: {len(amostra.columns)}")
for coluna in amostra.columns:
    print(repr(coluna))

display(amostra)

Colunas encontradas: 21
'Sigla ICAO Empresa Aérea'
'Empresa Aérea'
'Número Voo'
'Código DI'
'Código Tipo Linha'
'Modelo Equipamento'
'Número de Assentos'
'Sigla ICAO Aeroporto Origem'
'Descrição Aeroporto Origem'
'Partida Prevista'
'Partida Real'
'Sigla ICAO Aeroporto Destino'
'Descrição Aeroporto Destino'
'Chegada Prevista'
'Chegada Real'
'Situação Voo'
'Justificativa'
'Referência'
'Situação Partida'
'Situação Chegada'
'Codeshare'


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,Partida Real,Sigla ICAO Aeroporto Destino,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,Codeshare
0,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B788,295,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,01/01/2021 23:35,01/01/2021 23:29,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",02/01/2021 08:10,02/01/2021 07:56,REALIZADO,,01/01/2021 00:00:00,Antecipado,Antecipado,
1,AAL,"AMERICAN AIRLINES, INC.",0905,0,I,B788,295,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",01/01/2021 23:40,01/01/2021 23:34,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,02/01/2021 07:55,02/01/2021 07:45,REALIZADO,,01/01/2021 00:00:00,Antecipado,Antecipado,
2,AAL,"AMERICAN AIRLINES, INC.",0906,0,I,B77W,318,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,01/01/2021 00:05,01/01/2021 23:54,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",01/01/2021 08:35,02/01/2021 08:07,REALIZADO,,01/01/2021 00:00:00,Atraso > 240,Atraso > 240,
3,AAL,"AMERICAN AIRLINES, INC.",0929,0,I,B77W,318,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",01/01/2021 00:05,01/01/2021 23:57,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,01/01/2021 09:30,02/01/2021 08:12,REALIZADO,,01/01/2021 00:00:00,Atraso > 240,Atraso > 240,
4,AAL,"AMERICAN AIRLINES, INC.",0962,0,I,B789,295,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,01/01/2021 23:20,01/01/2021 23:18,KDFW,DALLAS-FORT WORTH INTERNATIONAL AIRPORT - DALL...,02/01/2021 09:50,02/01/2021 09:35,REALIZADO,,01/01/2021 00:00:00,Antecipado,Antecipado,


## 4. Conferir os cabeçalhos de todos os meses

Nesta tabela, `colunas_adicionais` e `colunas_ausentes` comparam cada arquivo aos
**20 campos do contrato**. Já `igual_a_janeiro` compara nomes e ordem com janeiro.
`Codeshare` é registrada como variação de esquema, não como erro do voo. Campos
obrigatórios do esquema ausentes, campos adicionais desconhecidos ou cabeçalhos
repetidos interrompem a execução para investigação.

In [4]:
cabecalho_janeiro = amostra.columns.tolist()
cabecalhos_por_arquivo = {}
verificacoes = []

for arquivo in arquivos:
    with arquivo.open(encoding="utf-8-sig", newline="") as entrada:
        cabecalho = next(csv.reader(entrada, delimiter=";", strict=True), [])

    if len(cabecalho) != len(set(cabecalho)):
        raise ValueError(f"Cabeçalho com nomes repetidos: {arquivo.name}")

    adicionais = [c for c in cabecalho if c not in colunas_comuns]
    ausentes = [c for c in colunas_comuns if c not in cabecalho]
    desconhecidas = [c for c in adicionais if c != "Codeshare"]

    if ausentes or desconhecidas:
        raise ValueError(
            f"Esquema inesperado em {arquivo.name}: "
            f"ausentes={ausentes}; adicionais desconhecidas={desconhecidas}."
        )

    cabecalhos_por_arquivo[arquivo.name] = cabecalho
    verificacoes.append({
        "arquivo": arquivo.name,
        "quantidade_colunas": len(cabecalho),
        "igual_a_janeiro": cabecalho == cabecalho_janeiro,
        "colunas_adicionais": adicionais,
        "colunas_ausentes": ausentes,
        "codeshare_presente": "Codeshare" in cabecalho,
    })

conferencia_cabecalhos = pd.DataFrame(verificacoes)
display(conferencia_cabecalhos)

,arquivo,quantidade_colunas,igual_a_janeiro,colunas_adicionais,colunas_ausentes,codeshare_presente
0,VRA_2021_01.csv,21,True,[Codeshare],[],True
1,VRA_2021_02.csv,21,True,[Codeshare],[],True
2,VRA_2021_03.csv,21,True,[Codeshare],[],True
3,VRA_2021_04.csv,20,False,[],[],False
4,VRA_2021_05.csv,20,False,[],[],False
5,VRA_2021_06.csv,20,False,[],[],False
6,VRA_2021_07.csv,21,True,[Codeshare],[],True
7,VRA_2021_08.csv,20,False,[],[],False
8,VRA_2021_09.csv,21,True,[Codeshare],[],True
9,VRA_2021_10.csv,20,False,[],[],False


## 5. Investigar o preenchimento de Codeshare

No recorte do projeto, o grupo informou que 2019 e 2020 possuem os 20 campos
comuns. Em 2021, a conferência identificou `Codeshare` em sete meses: janeiro,
fevereiro, março, julho, setembro, novembro e dezembro.

Sua presença é uma **variação de esquema**, não prova de deslocamento de colunas
nem motivo de rejeição. Contamos os valores preenchidos sem alterar o conteúdo:
`strip()` é usado somente na máscara desta contagem. Os exemplos mantêm o texto
original. Meses sem a coluna não recebem uma contagem artificial de zero.

In [5]:
resumo_codeshare = []

for arquivo in arquivos:
    if "Codeshare" not in cabecalhos_por_arquivo[arquivo.name]:
        continue

    dados_codeshare = pd.read_csv(
        arquivo,
        sep=";",
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
        skip_blank_lines=False,
        usecols=["Codeshare"],
    )

    coluna = dados_codeshare["Codeshare"]
    preenchidos = coluna.fillna("").str.strip().ne("")

    resumo_codeshare.append({
        "arquivo": arquivo.name,
        "registros": len(coluna),
        "codeshare_preenchido": int(preenchidos.sum()),
        "exemplos": coluna[preenchidos].drop_duplicates().head(5).tolist(),
    })

resumo_codeshare = pd.DataFrame(
    resumo_codeshare,
    columns=["arquivo", "registros", "codeshare_preenchido", "exemplos"],
)
display(resumo_codeshare)
print("Total de registros com Codeshare preenchido:",
      int(resumo_codeshare["codeshare_preenchido"].sum()))

Total de registros com Codeshare preenchido: 112479


,arquivo,registros,codeshare_preenchido,exemplos
0,VRA_2021_01.csv,61920,23061,"[ETD/4375, ETD/4399, GLO/5007, KLM/2411, KLM/2..."
1,VRA_2021_02.csv,46078,18763,"[ETD/4375, ETD/4399, GLO/5007, KLM/2411, GLO/5..."
2,VRA_2021_03.csv,41701,19217,"[ETD/4375, ETD/4399, GLO/5007, KLM/2411, GLO/5..."
3,VRA_2021_07.csv,55555,15357,"[ETD/4375, ETD/4399, GLO/5007, KLM/2411, KLM/2..."
4,VRA_2021_09.csv,58362,15577,"[ETD/4375, ETD/4399, GLO/5004, KLM/2412, KLM/2..."
5,VRA_2021_11.csv,66649,4219,"[JBU/4436, JBU/4437, JBU/4436 início: 27/03/20..."
6,VRA_2021_12.csv,74151,16285,"[JBU/4436, JBU/4437, JBU/4451, AZU/5340, AZU/5..."


## 6. Extrair os registros e acrescentar a proveniência

`pd.concat` reúne os meses e mantém todas as colunas da fonte. Nos meses sem
`Codeshare`, a coluna recebe um valor ausente na tabela reunida. A flag
`codeshare_presente_no_arquivo` distingue essa situação de um campo existente e
vazio. Os demais metadados permitem localizar cada ocorrência no arquivo original.

`linha_origem` conta o **registro lógico** do CSV, começando em 2 após o cabeçalho.
Um campo entre aspas pode conter quebras de linha, portanto esse número nem sempre
corresponde à linha física do arquivo. Não há limpeza ou deduplicação aqui.

In [6]:
partes = []

for arquivo in arquivos:
    df_mes = pd.read_csv(
        arquivo,
        sep=";",
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
        skip_blank_lines=False,
    )

    df_mes["codeshare_presente_no_arquivo"] = (
        "Codeshare" in cabecalhos_por_arquivo[arquivo.name]
    )
    df_mes["arquivo_origem"] = arquivo.name
    df_mes["linha_origem"] = range(2, len(df_mes) + 2)
    df_mes["ano_origem"] = ano_origem
    partes.append(df_mes)

df_bruta = pd.concat(partes, ignore_index=True, sort=False)
del partes, df_mes

if df_bruta.duplicated(subset=["arquivo_origem", "linha_origem"]).any():
    raise ValueError("Existem identificadores de origem repetidos na extração.")

print(f"Total de registros: {len(df_bruta):,}")
print(f"Total de colunas, incluindo metadados: {len(df_bruta.columns)}")
display(
    df_bruta.groupby("arquivo_origem", sort=True)
    .size()
    .rename("quantidade_registros")
    .reset_index()
)

Total de registros: 632,148
Total de colunas, incluindo metadados: 25


,arquivo_origem,quantidade_registros
0,VRA_2021_01.csv,61920
1,VRA_2021_02.csv,46078
2,VRA_2021_03.csv,41701
3,VRA_2021_04.csv,30224
4,VRA_2021_05.csv,37341
5,VRA_2021_06.csv,42512
6,VRA_2021_07.csv,55555
7,VRA_2021_08.csv,56367
8,VRA_2021_09.csv,58362
9,VRA_2021_10.csv,61288


## 7. Validar a largura dos registros e as contagens

Conferimos cada registro com `csv.reader`, respeitando aspas e quebras de linha
internas. A quantidade de campos precisa coincidir com o cabeçalho daquele mês,
e as contagens precisam coincidir com a extração feita pelo Pandas. Isso valida a
estrutura e as contagens; não comprova ainda a validade de datas e códigos.

In [7]:
contagens_pandas = df_bruta.groupby("arquivo_origem").size()
inventario = []

for arquivo in arquivos:
    with arquivo.open(encoding="utf-8-sig", newline="") as entrada:
        leitor = csv.reader(entrada, delimiter=";", strict=True)
        cabecalho = next(leitor)
        if cabecalho != cabecalhos_por_arquivo[arquivo.name]:
            raise ValueError(f"Cabeçalho alterado durante a leitura: {arquivo.name}")

        quantidade_campos = len(cabecalho)
        quantidade_registros = 0

        for linha_origem, registro in enumerate(leitor, start=2):
            if len(registro) != quantidade_campos:
                raise ValueError(
                    f"{arquivo.name}, registro {linha_origem}: "
                    f"esperados {quantidade_campos} campos, "
                    f"encontrados {len(registro)}."
                )
            quantidade_registros += 1

    quantidade_pandas = int(contagens_pandas[arquivo.name])
    if quantidade_registros != quantidade_pandas:
        raise ValueError(
            f"Contagem divergente em {arquivo.name}: "
            f"CSV={quantidade_registros}, Pandas={quantidade_pandas}."
        )


    sha256 = sha256_arquivo(arquivo)
    if sha256 != hashes_entrada[arquivo.name]:
        raise ValueError(f"Arquivo alterado durante a extração: {arquivo.name}")

    inventario.append({
        "arquivo": arquivo.name,
        "mes": int(arquivo.stem[-2:]),
        "bytes": arquivo.stat().st_size,
        "sha256": sha256,
        "colunas_originais": quantidade_campos,
        "registros_csv": quantidade_registros,
        "registros_pandas": quantidade_pandas,
    })

inventario = pd.DataFrame(inventario)
display(inventario)
print("Conferência estrutural e de contagens concluída.")

Conferência estrutural e de contagens concluída.


,arquivo,mes,bytes,sha256,colunas_originais,registros_csv,registros_pandas
0,VRA_2021_01.csv,1,18281868,6ee26c0fef40bf502f0557abd4d321a81ed0e9abb0c101...,21,61920,61920
1,VRA_2021_02.csv,2,13667479,693a2bacdad0a3cedaa6128c3cc10bb0691ccf23958c5e...,21,46078,46078
2,VRA_2021_03.csv,3,12444242,b590ba0323cbd37ef50f9272cf067ffe4176607516222b...,21,41701,41701
3,VRA_2021_04.csv,4,8455519,a55b75cceb5e90fa8f8d8ed0b7910b8dfcb2c45f9dddde...,20,30224,30224
4,VRA_2021_05.csv,5,10464126,49907e9546d02fd3cbbdcc5dc62bb8d64130ce91c03f05...,20,37341,37341
5,VRA_2021_06.csv,6,11928409,babaf8882b29fa3b89a83df226af6385519fbe222ae335...,20,42512,42512
6,VRA_2021_07.csv,7,16716773,fa1a34e334ed3faa5ad5d9bc1e7c39364744b68a4b8fd8...,21,55555,55555
7,VRA_2021_08.csv,8,15837098,4dd71d230cbe058fde0a645fac3897fbd44a33cd61326a...,20,56367,56367
8,VRA_2021_09.csv,9,17466130,f25fa7df2051735f0b09046f6242f34e9abb435dbf3dfe...,21,58362,58362
9,VRA_2021_10.csv,10,17231843,eb18165d34365084e57c3f6a7f6cbbad9428f06c20f906...,20,61288,61288


## 8. Registrar a variação de esquema em auditoria

A tabela abaixo registra a ocorrência por arquivo, sem rejeitar os voos nem
eliminar a coluna. O tratamento consiste em preservar os valores na base bruta e
documentar a extensão do esquema de 2021. Os 20 campos do contrato continuam sendo
a referência comum para integração e investigação de duplicatas.

A auditoria permanece em memória nesta etapa; sua exportação será preparada junto
com as demais evidências do pipeline.

In [8]:
auditoria_esquema = resumo_codeshare.copy()
auditoria_esquema["ano_origem"] = ano_origem
auditoria_esquema["tipo_ocorrencia"] = "VARIACAO_ESQUEMA_CODESHARE"
auditoria_esquema["decisao"] = (
    "Preservar na base bruta e auditar; manter os 20 campos comuns do contrato"
)

display(
    auditoria_esquema[
        ["arquivo", "ano_origem", "tipo_ocorrencia", "registros",
         "codeshare_preenchido", "decisao"]
    ]
)

,arquivo,ano_origem,tipo_ocorrencia,registros,codeshare_preenchido,decisao
0,VRA_2021_01.csv,2021,VARIACAO_ESQUEMA_CODESHARE,61920,23061,Preservar na base bruta e auditar; manter os 2...
1,VRA_2021_02.csv,2021,VARIACAO_ESQUEMA_CODESHARE,46078,18763,Preservar na base bruta e auditar; manter os 2...
2,VRA_2021_03.csv,2021,VARIACAO_ESQUEMA_CODESHARE,41701,19217,Preservar na base bruta e auditar; manter os 2...
3,VRA_2021_07.csv,2021,VARIACAO_ESQUEMA_CODESHARE,55555,15357,Preservar na base bruta e auditar; manter os 2...
4,VRA_2021_09.csv,2021,VARIACAO_ESQUEMA_CODESHARE,58362,15577,Preservar na base bruta e auditar; manter os 2...
5,VRA_2021_11.csv,2021,VARIACAO_ESQUEMA_CODESHARE,66649,4219,Preservar na base bruta e auditar; manter os 2...
6,VRA_2021_12.csv,2021,VARIACAO_ESQUEMA_CODESHARE,74151,16285,Preservar na base bruta e auditar; manter os 2...


## 9. Investigar duplicatas sem remover registros

Comparamos os **20 valores brutos** do contrato em todos os meses, sem incluir
metadados nem aplicar `strip`, conversão de datas ou mudança de maiúsculas.
`keep=False` identifica todas as ocorrências de grupos repetidos; `keep="first"`
conta apenas as ocorrências excedentes.

Em cada grupo, também investigamos diferenças em `Codeshare`. Um valor ausente
por falta da coluna, um texto vazio e um texto preenchido podem contar como
valores distintos. Uma variação exige interpretação antes da exclusão analítica;
ela não é automaticamente um erro nem autoriza mudar o contrato. **Esta célula
apenas diagnostica: `df_bruta` permanece completa.**

In [9]:
mascara_duplicadas = df_bruta.duplicated(
    subset=colunas_comuns,
    keep=False,
)
candidatas_duplicatas = df_bruta.loc[mascara_duplicadas].copy()

quantidade_excedentes = int(
    df_bruta.duplicated(subset=colunas_comuns, keep="first").sum()
)

comparacao_codeshare = (
    candidatas_duplicatas
    .groupby(colunas_comuns, dropna=False, sort=False)["Codeshare"]
    .nunique(dropna=False)
    .rename("quantidade_valores_codeshare")
    .reset_index()
)
variacoes_codeshare = comparacao_codeshare.loc[
    comparacao_codeshare["quantidade_valores_codeshare"] > 1
].copy()

resumo_duplicatas = pd.DataFrame([{
    "ano_origem": ano_origem,
    "registros_brutos": len(df_bruta),
    "registros_envolvidos": len(candidatas_duplicatas),
    "grupos_repetidos": len(comparacao_codeshare),
    "ocorrencias_excedentes": quantidade_excedentes,
    "grupos_com_variacao_codeshare": len(variacoes_codeshare),
}])

print("Registros envolvidos em repetições:", len(candidatas_duplicatas))
print("Grupos repetidos:", len(comparacao_codeshare))
print("Ocorrências excedentes:", quantidade_excedentes)
print("Grupos com variação em Codeshare:", len(variacoes_codeshare))
display(resumo_duplicatas)

if not variacoes_codeshare.empty:
    auditoria_variacoes_codeshare = candidatas_duplicatas.merge(
        variacoes_codeshare,
        on=colunas_comuns,
        how="inner",
        validate="many_to_one",
    )
    display(
        auditoria_variacoes_codeshare[
            ["arquivo_origem", "linha_origem", "Número Voo", "Partida Prevista",
             "Codeshare", "codeshare_presente_no_arquivo"]
        ].head(20)
    )
    print("Revise estas variações antes de excluir ocorrências repetidas.")
else:
    auditoria_variacoes_codeshare = candidatas_duplicatas.iloc[0:0].copy()
    print("Nenhum grupo repetido apresentou variação em Codeshare.")

Registros envolvidos em repetições: 0
Grupos repetidos: 0
Ocorrências excedentes: 0
Grupos com variação em Codeshare: 0
Nenhum grupo repetido apresentou variação em Codeshare.


,ano_origem,registros_brutos,registros_envolvidos,grupos_repetidos,ocorrencias_excedentes,grupos_com_variacao_codeshare
0,2021,632148,0,0,0,0


## 10. Separar duplicatas exatas com auditoria

Comparamos os 20 campos originais antes da padronização.
Preservamos a primeira ocorrência por arquivo e registro de origem.
Os excedentes ficam separados, com referência à ocorrência preservada.
A base bruta permanece intacta.

In [10]:
df_ordenada = df_bruta.sort_values(
    ["arquivo_origem", "linha_origem"],
    kind="stable",
)

mascara_excedentes = df_ordenada.duplicated(
    subset=colunas_comuns,
    keep="first",
)

df_sem_duplicatas = df_ordenada.loc[~mascara_excedentes].copy()

# Localização da ocorrência preservada de cada conjunto de valores.
origens_preservadas = df_sem_duplicatas[
    colunas_comuns + ["arquivo_origem", "linha_origem"]
].rename(columns={
    "arquivo_origem": "arquivo_preservado",
    "linha_origem": "linha_preservada",
})

df_duplicatas = df_ordenada.loc[mascara_excedentes].merge(
    origens_preservadas,
    on=colunas_comuns,
    how="left",
    validate="many_to_one",
)

if len(df_bruta) != len(df_sem_duplicatas) + len(df_duplicatas):
    raise ValueError("Falha na reconciliação da deduplicação.")

print("Registros brutos:", len(df_bruta))
print("Excedentes separados:", len(df_duplicatas))
print("Registros após deduplicação:", len(df_sem_duplicatas))

Registros brutos: 632148
Excedentes separados: 0
Registros após deduplicação: 632148


## 11. Renomear e padronizar os textos em uma nova camada

Criamos `df_padronizada` a partir dos registros deduplicados. `df_bruta` e
`df_sem_duplicatas` continuam com os valores originais. Os 20 campos passam aos
nomes canônicos, vazios viram nulos e espaços externos são removidos.

Códigos e situações são convertidos para maiúsculas. Nomes de companhias,
descrições de aeroportos e justificativas preservam sua grafia. `numero_voo` e
`codigo_di` continuam texto. `Codeshare` permanece na camada bruta e nas auditorias,
fora dos atributos comuns da camada tratada.

In [11]:
mapa_colunas = {
    "Sigla ICAO Empresa Aérea": "companhia_icao",
    "Empresa Aérea": "companhia_nome",
    "Número Voo": "numero_voo",
    "Código DI": "codigo_di",
    "Código Tipo Linha": "tipo_linha",
    "Modelo Equipamento": "modelo_equipamento",
    "Número de Assentos": "numero_assentos",
    "Sigla ICAO Aeroporto Origem": "aeroporto_origem",
    "Descrição Aeroporto Origem": "descricao_origem",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Sigla ICAO Aeroporto Destino": "aeroporto_destino",
    "Descrição Aeroporto Destino": "descricao_destino",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Justificativa": "justificativa",
    "Referência": "referencia",
    "Situação Partida": "situacao_partida",
    "Situação Chegada": "situacao_chegada",
}
colunas_canonicas = list(mapa_colunas.values())
metadados = ["arquivo_origem", "linha_origem", "ano_origem"]
campos_maiusculos = [
    "companhia_icao", "codigo_di", "tipo_linha", "modelo_equipamento",
    "aeroporto_origem", "aeroporto_destino", "situacao_voo",
    "situacao_partida", "situacao_chegada",
]


def padronizar_textos(base):
    saida = base[colunas_comuns + metadados].copy()
    saida.rename(columns=mapa_colunas, inplace=True)
    for campo in colunas_canonicas:
        texto = saida[campo].astype("string").str.strip()
        saida[campo] = texto.mask(texto.eq(""), pd.NA)
    for campo in campos_maiusculos:
        saida[campo] = saida[campo].str.upper()
    return saida


df_padronizada = padronizar_textos(df_sem_duplicatas)
mascara_pos_trim_upper = df_padronizada.duplicated(
    subset=colunas_canonicas, keep=False
)
duplicatas_pos_trim_upper = df_padronizada.loc[mascara_pos_trim_upper].copy()
excedentes_pos_trim_upper = int(df_padronizada.duplicated(
    subset=colunas_canonicas, keep="first"
).sum())

print("Registros na camada padronizada:", len(df_padronizada))
print("Excedentes potenciais somente após TRIM/UPPER:", excedentes_pos_trim_upper)
print("Essas repetições são auditadas; não são removidas automaticamente.")
display(df_padronizada[["companhia_icao", "numero_voo", "partida_prevista", "referencia"]].head())

Registros na camada padronizada: 632148
Excedentes potenciais somente após TRIM/UPPER: 0
Essas repetições são auditadas; não são removidas automaticamente.


,companhia_icao,numero_voo,partida_prevista,referencia
0,AAL,0904,01/01/2021 23:35,01/01/2021 00:00:00
1,AAL,0905,01/01/2021 23:40,01/01/2021 00:00:00
2,AAL,0906,01/01/2021 00:05,01/01/2021 00:00:00
3,AAL,0929,01/01/2021 00:05,01/01/2021 00:00:00
4,AAL,0962,01/01/2021 23:20,01/01/2021 00:00:00


## 12. Converter datas e assentos sem corrigir valores inválidos

As quatro datas operacionais exigem `DD/MM/AAAA HH:MM`. `referencia` aceita
`AAAA-MM-DD` ou `DD/MM/AAAA 00:00:00` e vira uma data sem horário. As funções
validam calendário e horário, sem troca automática de dia/mês nem conversão de fuso.

Assentos devem conter somente dígitos e caber no inteiro PostgreSQL não negativo
(0 a 2.147.483.647). Valores opcionais inválidos tornam-se nulos e serão auditados,
sem rejeitar o voo por si só. As datas usam precisão de microssegundos no Pandas
para preservar anos válidos até 9999, sem impor o limite do datetime64 em nanossegundos.

In [12]:
PADRAO_DATAHORA = re.compile(r"[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}")
PADRAO_DATA_ISO = re.compile(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
PADRAO_REFERENCIA_BR = re.compile(r"[0-9]{2}/[0-9]{2}/[0-9]{4} 00:00:00")


def datahora_segura(valor):
    if pd.isna(valor) or not PADRAO_DATAHORA.fullmatch(valor):
        return None
    try:
        return datetime(int(valor[6:10]), int(valor[3:5]), int(valor[0:2]),
                        int(valor[11:13]), int(valor[14:16]))
    except ValueError:
        return None


def referencia_segura(valor):
    if pd.isna(valor):
        return None
    try:
        if PADRAO_DATA_ISO.fullmatch(valor):
            return date(int(valor[0:4]), int(valor[5:7]), int(valor[8:10]))
        if PADRAO_REFERENCIA_BR.fullmatch(valor):
            return date(int(valor[6:10]), int(valor[3:5]), int(valor[0:2]))
    except ValueError:
        return None
    return None


def assentos_seguros(valor):
    if pd.isna(valor) or not re.fullmatch(r"[0-9]+", valor):
        return None
    digitos = valor.lstrip("0") or "0"
    if len(digitos) > 10:
        return None
    numero = int(digitos)
    return numero if numero <= 2147483647 else None


def converter_tipos(base):
    saida = base.copy()
    for campo in ["partida_prevista", "partida_real", "chegada_prevista", "chegada_real"]:
        saida[campo] = pd.Series(
            [datahora_segura(valor) for valor in base[campo]],
            index=base.index, dtype="datetime64[us]",
        )
    saida["referencia"] = pd.Series(
        [referencia_segura(valor) for valor in base["referencia"]],
        index=base.index, dtype=object,
    )
    saida["numero_assentos"] = pd.Series(
        [assentos_seguros(valor) for valor in base["numero_assentos"]],
        index=base.index, dtype="Int32",
    )
    return saida


df_convertida = converter_tipos(df_padronizada)
perfil_conversoes = pd.DataFrame([
    {
        "campo": campo,
        "preenchidos_na_fonte": int(df_padronizada[campo].notna().sum()),
        "nulos_na_fonte": int(df_padronizada[campo].isna().sum()),
        "preenchidos_nao_convertidos": int((
            df_padronizada[campo].notna() & df_convertida[campo].isna()
        ).sum()),
    }
    for campo in ["partida_prevista", "partida_real", "chegada_prevista",
                  "chegada_real", "referencia", "numero_assentos"]
])
display(perfil_conversoes)

,campo,preenchidos_na_fonte,nulos_na_fonte,preenchidos_nao_convertidos
0,partida_prevista,606637,25511,0
1,partida_real,606009,26139,0
2,chegada_prevista,606637,25511,0
3,chegada_real,606007,26141,0
4,referencia,632148,0,0
5,numero_assentos,632148,0,0


## 13. Separar rejeições e auditar valores opcionais

Somente os cinco campos obrigatórios determinam a rejeição: companhia, aeroporto
de origem, aeroporto de destino, partida prevista e chegada prevista. A distinção
entre data ausente e data inválida é preservada. Todos os motivos são registrados,
mas cada ocorrência conta uma única vez na quantidade de rejeitados.

As rejeições mantêm os campos brutos, inclusive `Codeshare`, e a proveniência.
Anomalias opcionais são registradas por campo: uma linha pode gerar mais de uma.
A auditoria de `referencia` fora de 2021 não causa exclusão automática.

In [13]:
def motivos_obrigatorios(textos, convertida):
    regras = {
        "COMPANHIA_AUSENTE": convertida["companhia_icao"].isna(),
        "AEROPORTO_ORIGEM_AUSENTE": convertida["aeroporto_origem"].isna(),
        "AEROPORTO_DESTINO_AUSENTE": convertida["aeroporto_destino"].isna(),
    }
    for campo, prefixo in [("partida_prevista", "PARTIDA_PREVISTA"),
                           ("chegada_prevista", "CHEGADA_PREVISTA")]:
        regras[prefixo + "_AUSENTE"] = textos[campo].isna()
        regras[prefixo + "_INVALIDA"] = textos[campo].notna() & convertida[campo].isna()
    motivos = [[] for _ in range(len(convertida))]
    for motivo, mascara in regras.items():
        for posicao in np.flatnonzero(mascara.to_numpy(dtype=bool)):
            motivos[posicao].append(motivo)
    return pd.Series(motivos, index=convertida.index, name="motivos_rejeicao")


def auditar_opcionais(bruta, textos, convertida):
    partes_anomalias = []
    for original, canonico in [
        ("Partida Real", "partida_real"),
        ("Chegada Real", "chegada_real"),
        ("Número de Assentos", "numero_assentos"),
        ("Referência", "referencia"),
    ]:
        mascara = textos[canonico].notna() & convertida[canonico].isna()
        ocorrencias = bruta.loc[mascara, metadados].copy()
        ocorrencias["campo"] = original
        ocorrencias["valor_original"] = bruta.loc[mascara, original]
        ocorrencias["motivo"] = "VALOR_OPCIONAL_INVALIDO"
        partes_anomalias.append(ocorrencias)

    fora_ano = convertida["referencia"].map(
        lambda valor: valor is not None and valor.year != ano_origem
    )
    ocorrencias = bruta.loc[fora_ano, metadados].copy()
    ocorrencias["campo"] = "Referência"
    ocorrencias["valor_original"] = bruta.loc[fora_ano, "Referência"]
    ocorrencias["motivo"] = "REFERENCIA_FORA_DE_2021"
    partes_anomalias.append(ocorrencias)
    return pd.concat(partes_anomalias, ignore_index=True)


motivos_rejeicao = motivos_obrigatorios(df_padronizada, df_convertida)
mascara_rejeicao = motivos_rejeicao.map(bool)
df_rejeicoes = df_sem_duplicatas.loc[mascara_rejeicao].copy()
df_rejeicoes["motivos_rejeicao"] = motivos_rejeicao.loc[mascara_rejeicao]
df_tratada = df_convertida.loc[~mascara_rejeicao].copy()
vra_anomalias = auditar_opcionais(df_sem_duplicatas, df_padronizada, df_convertida)

motivos_detalhados = (
    df_rejeicoes[metadados + ["motivos_rejeicao"]]
    .explode("motivos_rejeicao")
    .rename(columns={"motivos_rejeicao": "motivo"})
)
contagem_motivos = (
    motivos_detalhados.groupby("motivo").size()
    .rename("quantidade").reset_index()
    .sort_values(["quantidade", "motivo"], ascending=[False, True])
)

print("Registros rejeitados:", len(df_rejeicoes))
print("Registros elegíveis:", len(df_tratada))
print("Ocorrências na auditoria de anomalias:", len(vra_anomalias))
display(contagem_motivos)

Registros rejeitados: 25511
Registros elegíveis: 606637
Ocorrências na auditoria de anomalias: 0


,motivo,quantidade
0,CHEGADA_PREVISTA_AUSENTE,25511
1,PARTIDA_PREVISTA_AUSENTE,25511


## 14. Calcular diferenças, atrasos e flags de 24 horas

A diferença preserva o sinal: real menos previsto, em minutos. O atraso é essa
diferença limitada inferiormente a zero. Sem horário real, as duas medidas ficam
nulas. Usamos as datas completas, incluindo cruzamentos de dia, mês e ano.

Diferenças absolutas **estritamente maiores** que 1.440 minutos são sinalizadas,
sem alteração do dado ou exclusão do voo. Sem diferença calculável, a flag é falsa.
Os registros cancelados e não informados permanecem em categorias separadas.

In [14]:
def calcular_metricas(base):
    saida = base.copy()
    for evento in ["partida", "chegada"]:
        diferenca = (
            (saida[f"{evento}_real"] - saida[f"{evento}_prevista"])
            .dt.total_seconds().div(60).astype("Int64")
        )
        saida[f"diferenca_{evento}_min"] = diferenca
        saida[f"atraso_{evento}_min"] = diferenca.clip(lower=0)
        saida[f"{evento}_atipica_flag"] = diferenca.abs().gt(1440).fillna(False).astype(bool)
    return saida


df_tratada = calcular_metricas(df_tratada)
situacao_voo = (
    df_tratada.groupby("situacao_voo", dropna=False).size()
    .rename("quantidade").reset_index()
    .sort_values("quantidade", ascending=False)
)
mascara_atipicos = df_tratada["partida_atipica_flag"] | df_tratada["chegada_atipica_flag"]
vra_atipicos = df_tratada.loc[
    mascara_atipicos,
    metadados + ["numero_voo", "partida_prevista", "partida_real", "chegada_prevista",
                 "chegada_real", "diferenca_partida_min", "diferenca_chegada_min"],
].copy()
mascara_fora_ano = df_tratada["partida_prevista"].dt.year.ne(ano_origem)
partidas_previstas_fora_ano = df_tratada.loc[mascara_fora_ano].copy()

print("Partidas com diferença absoluta >24h:", int(df_tratada["partida_atipica_flag"].sum()))
print("Chegadas com diferença absoluta >24h:", int(df_tratada["chegada_atipica_flag"].sum()))
display(situacao_voo)
display(vra_atipicos.head(5))

Partidas com diferença absoluta >24h: 371
Chegadas com diferença absoluta >24h: 373


,situacao_voo,quantidade
2,REALIZADO,580498
0,CANCELADO,25962
1,NÃO INFORMADO,177


,arquivo_origem,linha_origem,ano_origem,numero_voo,partida_prevista,partida_real,chegada_prevista,chegada_real,diferenca_partida_min,diferenca_chegada_min
24894,VRA_2021_01.csv,24896,2021,0873,2021-01-03 17:30:00,2021-01-02 17:24:00,2021-01-04 00:35:00,2021-01-03 00:21:00,-1446,-1454
24905,VRA_2021_01.csv,24907,2021,0873,2021-01-04 17:30:00,2021-01-03 17:19:00,2021-01-05 00:35:00,2021-01-04 00:28:00,-1451,-1447
24935,VRA_2021_01.csv,24937,2021,0873,2021-01-07 17:30:00,2021-01-06 17:29:00,2021-01-08 00:35:00,2021-01-07 01:13:00,-1441,-1402
24946,VRA_2021_01.csv,24948,2021,0873,2021-01-08 17:30:00,2021-01-07 17:25:00,2021-01-09 00:35:00,2021-01-08 00:39:00,-1445,-1436
24957,VRA_2021_01.csv,24959,2021,0873,2021-01-09 17:30:00,2021-01-08 17:25:00,2021-01-10 00:35:00,2021-01-09 00:22:00,-1445,-1453


## 15. Identificar conflitos de descrição e repetições após conversão

Um mesmo código ICAO pode aparecer com descrições diferentes. Registramos todas
as variantes e suas contagens, sem escolher arbitrariamente uma para a dimensão
compartilhada. Os aeroportos de origem e destino são analisados juntos.

Também contamos repetições pelos 20 atributos já convertidos, como no resumo do
ELT de Murilo. Esse indicador é distinto do diagnóstico apenas após TRIM/UPPER:
conversões de datas e números também podem aproximar valores. Nenhum desses
indicadores provoca uma segunda deduplicação.

In [15]:
def conflitos_descricao(base):
    partes = []
    for tipo, codigo, descricao in [
        ("companhia", "companhia_icao", "companhia_nome"),
        ("aeroporto", "aeroporto_origem", "descricao_origem"),
        ("aeroporto", "aeroporto_destino", "descricao_destino"),
    ]:
        pares = base[[codigo, descricao]].rename(
            columns={codigo: "codigo", descricao: "descricao"}
        )
        pares["tipo"] = tipo
        partes.append(pares)
    pares = pd.concat(partes, ignore_index=True)
    quantidades = pares.groupby(["tipo", "codigo"])["descricao"].nunique(dropna=False)
    chaves = quantidades.loc[quantidades > 1].reset_index()[["tipo", "codigo"]]
    variantes = pares.merge(chaves, on=["tipo", "codigo"], how="inner", validate="many_to_one")
    return (
        variantes.groupby(["tipo", "codigo", "descricao"], dropna=False).size()
        .rename("ocorrencias").reset_index()
        .sort_values(["tipo", "codigo", "descricao"], na_position="last")
    )


vra_conflitos_codigo = conflitos_descricao(df_tratada)
excedentes_apos_padronizacao = int(df_convertida.duplicated(
    subset=colunas_canonicas, keep="first"
).sum())
duplicatas_apos_padronizacao = df_convertida.loc[
    df_convertida.duplicated(subset=colunas_canonicas, keep=False)
].copy()
qualidade_colunas = pd.DataFrame({
    "campo": df_convertida.columns,
    "quantidade_nulos": df_convertida.isna().sum().to_numpy(),
})
print("Códigos com descrições conflitantes:",
      len(vra_conflitos_codigo[["tipo", "codigo"]].drop_duplicates()))
print("Excedentes potenciais após conversão:", excedentes_apos_padronizacao)
display(vra_conflitos_codigo)

Códigos com descrições conflitantes: 9
Excedentes potenciais após conversão: 0


,tipo,codigo,descricao,ocorrencias
0,aeroporto,KAEX,"ALEXANDRIA INTERNATIONAL AIRPORT - ALEXANDRIA,...",5
1,aeroporto,KAEX,"ALEXANDRIA INTERNATIONAL AIRPORT - ALEXANDRIA,...",21
2,aeroporto,SWHP,OLHOS D´ÁGUA - ÁGUA BOA - MG - BRASIL,109
3,aeroporto,SWHP,OLHOS D´ÁGUA - ÁGUA BOA - MT - BRASIL,310
4,aeroporto,SYEC,EUGENE F. CORREIRA INTERNATIONAL - OGLE - - G...,14
5,aeroporto,SYEC,EUGENE F. CORREIRA INTERNATIONAL - OGLE - GUIANA,6
6,companhia,CQB,APUÍ TÁXI AÉREO LTDA.,102
7,companhia,CQB,APUÍ TÁXI AÉREO S/A,27
8,companhia,DTA,TAAG LINHAS AÉREAS DA ANGOLA (ANGOLA AIRLINES),26
9,companhia,DTA,TAAG LINHAS AÉREAS DE ANGOLA (ANGOLA AIRLINES),33


## 16. Validar a conciliação e consolidar os indicadores

A igualdade obrigatória é: **brutos = excedentes + rejeitados após deduplicação +
tratados**. Também conferimos cada mês, a separação dos conjuntos, os campos
obrigatórios, a preservação dos zeros à esquerda e a relação entre as métricas e
os horários. A reconciliação não conta motivos como se fossem novos registros.

Os resultados informados pelo ELT de Murilo são mostrados apenas para comparação.
Totais iguais são necessários, mas a comparação completa ainda exige os hashes
dos arquivos do parceiro e os valores por ocorrência de origem.

In [16]:
resumo_qualidade = pd.DataFrame([{
    "ano_origem": ano_origem,
    "bruta": len(df_bruta),
    "duplicatas_excedentes": len(df_duplicatas),
    "rejeicoes": len(df_rejeicoes),
    "tratada": len(df_tratada),
    "grupos_duplicados": len(comparacao_codeshare),
    "registros_envolvidos_duplicatas": len(candidatas_duplicatas),
    "duplicatas_apos_trim_upper": excedentes_pos_trim_upper,
    "duplicatas_apos_padronizacao": excedentes_apos_padronizacao,
    "partidas_atipicas": int(df_tratada["partida_atipica_flag"].sum()),
    "chegadas_atipicas": int(df_tratada["chegada_atipica_flag"].sum()),
    "partidas_reais_nulas": int(df_tratada["partida_real"].isna().sum()),
    "chegadas_reais_nulas": int(df_tratada["chegada_real"].isna().sum()),
    "ocorrencias_anomalias": len(vra_anomalias),
    "partidas_previstas_fora_do_ano": len(partidas_previstas_fora_ano),
    "codigos_com_conflito": len(vra_conflitos_codigo[["tipo", "codigo"]].drop_duplicates()),
    "arquivos_com_codeshare": int(conferencia_cabecalhos["codeshare_presente"].sum()),
    "codeshare_preenchido": int(resumo_codeshare["codeshare_preenchido"].sum()),
    "executado_em": datetime.now(timezone.utc).isoformat(),
}])

mensal = inventario[["arquivo", "registros_csv"]].rename(
    columns={"arquivo": "arquivo_origem", "registros_csv": "bruta"}
).copy()
for nome, base in [("duplicatas", df_duplicatas), ("rejeicoes", df_rejeicoes), ("tratada", df_tratada)]:
    contagem = base.groupby("arquivo_origem").size()
    mensal[nome] = mensal["arquivo_origem"].map(contagem).fillna(0).astype("int64")
mensal["conciliada"] = mensal["bruta"].eq(mensal["duplicatas"] + mensal["rejeicoes"] + mensal["tratada"])


def exigir(condicao, mensagem):
    if not bool(condicao):
        raise ValueError(mensagem)


exigir(len(df_bruta) == len(df_duplicatas) + len(df_rejeicoes) + len(df_tratada),
       "A conciliação anual falhou.")
exigir(mensal["conciliada"].all(), "A conciliação mensal falhou.")
exigir(not df_tratada[["companhia_icao", "aeroporto_origem", "aeroporto_destino",
                       "partida_prevista", "chegada_prevista"]].isna().any().any(),
       "A camada tratada contém campos obrigatórios nulos.")

chaves_tratada = pd.MultiIndex.from_frame(df_tratada[["arquivo_origem", "linha_origem"]])
chaves_rejeicoes = pd.MultiIndex.from_frame(df_rejeicoes[["arquivo_origem", "linha_origem"]])
chaves_duplicatas = pd.MultiIndex.from_frame(df_duplicatas[["arquivo_origem", "linha_origem"]])
exigir(chaves_tratada.is_unique, "A camada tratada tem origem repetida.")
exigir(chaves_tratada.intersection(chaves_rejeicoes).empty, "Há tratados também rejeitados.")
exigir(chaves_tratada.intersection(chaves_duplicatas).empty, "Há excedentes na camada tratada.")
exigir(chaves_rejeicoes.intersection(chaves_duplicatas).empty, "Há excedentes contados como rejeições.")

for evento in ["partida", "chegada"]:
    diferenca = df_tratada[f"diferenca_{evento}_min"]
    atraso = df_tratada[f"atraso_{evento}_min"]
    sem_real = df_tratada[f"{evento}_real"].isna()
    exigir(diferenca.isna().equals(sem_real), f"Nulos inconsistentes na diferença de {evento}.")
    exigir(atraso.isna().equals(sem_real), f"Nulos inconsistentes no atraso de {evento}.")
    exigir((atraso.dropna() >= 0).all(), f"Atraso negativo em {evento}.")
    minutos = diferenca.to_numpy(dtype="int64", na_value=np.iinfo("int64").min)
    intervalos = pd.Series(
        minutos.astype("timedelta64[m]").astype("timedelta64[us]"),
        index=df_tratada.index,
    )
    reconstruida = df_tratada[f"{evento}_real"] - intervalos
    exigir(reconstruida.loc[~sem_real].equals(df_tratada.loc[~sem_real, f"{evento}_prevista"]),
           f"A diferença não reconstrói o horário previsto de {evento}.")

exigir(df_padronizada["numero_voo"].equals(
    df_sem_duplicatas["Número Voo"].astype("string").str.strip().replace("", pd.NA)
), "O número do voo foi alterado além da padronização de espaços e vazios.")
exigir(all(sha256_arquivo(arquivo) == hashes_entrada[arquivo.name] for arquivo in arquivos),
       "Algum CSV original foi alterado.")

referencia_elt = pd.Series({"bruta": 632148, "duplicatas_excedentes": 0,
                            "rejeicoes": 25511, "tratada": 606637}, name="elt_informado")
comparativo_elt = resumo_qualidade.iloc[0][referencia_elt.index].rename("etl_medido").to_frame()
comparativo_elt["elt_informado"] = referencia_elt
comparativo_elt["totais_iguais"] = comparativo_elt["etl_medido"].eq(comparativo_elt["elt_informado"])
validacao_concluida = True

display(resumo_qualidade.T.rename(columns={0: "resultado"}))
display(mensal)
display(comparativo_elt)
print("Conciliações e verificações internas concluídas.")
print("Comparação dos hashes e dos valores por registro com o ELT: pendente.")

Conciliações e verificações internas concluídas.
Comparação dos hashes e dos valores por registro com o ELT: pendente.


,resultado
ano_origem,2021
bruta,632148
duplicatas_excedentes,0
rejeicoes,25511
tratada,606637
grupos_duplicados,0
registros_envolvidos_duplicatas,0
duplicatas_apos_trim_upper,0
duplicatas_apos_padronizacao,0
partidas_atipicas,371


,arquivo_origem,bruta,duplicatas,rejeicoes,tratada,conciliada
0,VRA_2021_01.csv,61920,0,3505,58415,True
1,VRA_2021_02.csv,46078,0,2486,43592,True
2,VRA_2021_03.csv,41701,0,2326,39375,True
3,VRA_2021_04.csv,30224,0,1250,28974,True
4,VRA_2021_05.csv,37341,0,1739,35602,True
5,VRA_2021_06.csv,42512,0,1680,40832,True
6,VRA_2021_07.csv,55555,0,2335,53220,True
7,VRA_2021_08.csv,56367,0,1517,54850,True
8,VRA_2021_09.csv,58362,0,1881,56481,True
9,VRA_2021_10.csv,61288,0,1860,59428,True


,etl_medido,elt_informado,totais_iguais
bruta,632148,632148,True
duplicatas_excedentes,0,0,True
rejeicoes,25511,25511,True
tratada,606637,606637,True


## 17. Exportar as evidências da execução

A pasta de saída fica ignorada pelo Git para evitar publicar acidentalmente a
base completa. Os arquivos incluem a camada tratada, rejeições com valores
originais, auditorias, hashes, contagens mensais, indicadores e um texto de
explicação. Listas de motivos são exportadas como JSON dentro do CSV.

Os CSVs originais já preservam a entrada. Para gerar também uma cópia tabular da
base bruta com metadados, habilite `EXPORTAR_BASE_BRUTA` na configuração.

In [17]:
arquivos_exportados = []


def exportar_tabela(nome, base, comprimir=True):
    destino = PASTA_SAIDA / (nome + (".csv.gz" if comprimir else ".csv"))
    tabela = base.copy()
    for campo in tabela.select_dtypes(include=["object"]).columns:
        if tabela[campo].map(lambda valor: isinstance(valor, list)).any():
            tabela[campo] = tabela[campo].map(
                lambda valor: json.dumps(valor, ensure_ascii=False) if isinstance(valor, list) else valor
            )
    tabela.to_csv(destino, index=False, encoding="utf-8", na_rep="",
                  compression="gzip" if comprimir else None)
    arquivos_exportados.append(destino)


if EXPORTAR_EVIDENCIAS:
    exigir(validacao_concluida, "Execute a validação antes de exportar.")
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    for nome, base in [
        ("vra_tratada", df_tratada),
        ("vra_rejeicoes", df_rejeicoes),
        ("vra_duplicatas_excluidas", df_duplicatas),
        ("vra_anomalias", vra_anomalias),
        ("vra_atipicos", vra_atipicos),
        ("duplicatas_pos_trim_upper", duplicatas_pos_trim_upper),
        ("duplicatas_apos_padronizacao", duplicatas_apos_padronizacao),
        ("partidas_previstas_fora_ano", partidas_previstas_fora_ano),
    ]:
        exportar_tabela(nome, base)
    for nome, base in [
        ("inventario_arquivos", inventario), ("resumo_qualidade", resumo_qualidade),
        ("mensal", mensal), ("motivos_rejeicao", contagem_motivos),
        ("situacao_voo", situacao_voo), ("vra_conflitos_codigo", vra_conflitos_codigo),
        ("auditoria_esquema", auditoria_esquema), ("qualidade_colunas", qualidade_colunas),
        ("perfil_conversoes", perfil_conversoes),
        ("comparativo_totais_elt_informados", comparativo_elt.reset_index(names="indicador")),
    ]:
        exportar_tabela(nome, base, comprimir=False)
    if EXPORTAR_BASE_BRUTA:
        exportar_tabela("vra_bruta", df_bruta)

    resumo_dict = json.loads(resumo_qualidade.to_json(orient="records"))[0]
    metadados_execucao = {
        "autor": "Arthur", "pipeline": "ETL", "ano_origem": ano_origem,
        "versao_contrato": "1.1", "inicio_utc": inicio_execucao,
        "fim_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0],
        "pandas": pd.__version__, "transformacoes": "Python/Pandas antes da carga",
        "resumo": resumo_dict,
        "csvs_originais_intactos": True,
        "comparacao_elt": "Totais informados comparados; hashes e valores por registro pendentes.",
        "carga_postgresql": "A etapa seguinte informa se a carga foi executada.",
        "modelagem_dimensional": "Pendente da integração do grupo.",
    }
    (PASTA_SAIDA / "metadados_execucao.json").write_text(
        json.dumps(metadados_execucao, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    texto_explicacao = f"""# ETL VRA/ANAC 2021 — execução local

Responsável: Arthur. Referências: contrato v1.1 e regras/saídas do ELT 2021 de Murilo Acioli.

O Python lê os CSVs como texto e compara os 20 campos brutos antes da padronização.
Codeshare permanece na base bruta e sua presença é auditada por arquivo.
As transformações, rejeições, métricas e conflitos são calculados em Python/Pandas.
A carga opcional recebe os dados já tratados no schema {SCHEMA_ETL}.

Reconciliação medida: {len(df_bruta)} brutos = {len(df_duplicatas)} excedentes +
{len(df_rejeicoes)} rejeitados + {len(df_tratada)} tratados.

Vazios tornam-se nulos; códigos e situações ficam em maiúsculas; descrições preservam a grafia.
Número do voo continua texto. Referência aceita ISO e data brasileira à meia-noite.
Somente problemas nos cinco campos obrigatórios provocam rejeição.
Opcionais inválidos são auditados. Todos os motivos de cada rejeição são preservados.
Diferenças usam minutos com sinal; atrasos são não negativos; sem real, métricas são nulas.
Diferenças absolutas acima de 24 horas são sinalizadas e preservadas, sem correção automática.
Não foi aplicada conversão de fuso. Conflitos de descrições são reportados sem escolher um nome.

Limites: dimensões e Fato compartilhadas ainda dependem da integração do grupo.
A igualdade dos totais com o ELT não comprova equivalência por registro.
Compare os hashes do inventário e a saída tratada do parceiro antes de concluir a conciliação.
O texto colado do ELT não incluía o corpo completo de suas funções SQL.
As regras foram implementadas pelo contrato e pelo comportamento documentado.

A etapa PostgreSQL permanece desativada por padrão. Veja status_carga_postgresql.json.
Os CSVs originais não foram sobrescritos. Evidências completas ficam em dados/processados/2021/etl/.
"""
    (PASTA_SAIDA / "LEIA_ME.md").write_text(texto_explicacao, encoding="utf-8")
    print(f"Evidências exportadas para: {PASTA_SAIDA}")
    print("Tabelas exportadas:", len(arquivos_exportados))
else:
    print("Exportação desativada na configuração.")

Evidências exportadas para: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/dados/processados/2021/etl
Tabelas exportadas: 18


## 18. Carregar a camada tratada no PostgreSQL — opcional

Esta etapa está **desativada por padrão**. Para executá-la, prepare o servidor e
o banco, configure as credenciais em `.env` ou variáveis de ambiente e habilite
`EXECUTAR_CARGA_POSTGRES` na configuração. O notebook não instala nem cria o banco.
No Colab, use variáveis de ambiente configuradas de forma privada.

A carga usa o schema exclusivo `etl_2021`, sem usar automaticamente o `PGSCHEMA`
do ETL de 2019. Tabelas existentes interrompem a carga, a menos que
`SUBSTITUIR_CARGA_POSTGRES=True`. A substituição se limita às tabelas listadas
deste pipeline, dentro de uma transação; nenhum schema inteiro é apagado.

SQL é usado apenas para criar/carregar tabelas e verificar contagens e chaves.
Todas as transformações já terminaram em Python. Esta etapa ainda precisa de
validação num PostgreSQL real; não é equivalente à criação das dimensões/Fato.

In [18]:
status_carga_postgresql = {"status": "nao_executada", "schema": SCHEMA_ETL}

if EXECUTAR_CARGA_POSTGRES:
    import os
    from dotenv import load_dotenv
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import URL, make_url
    from sqlalchemy import types as sat
    from sqlalchemy.dialects.postgresql import ARRAY

    exigir(validacao_concluida, "Valide as transformações antes da carga.")
    exigir(re.fullmatch(r"[a-z_][a-z0-9_]*", SCHEMA_ETL) is not None,
           "Nome de schema inválido.")
    exigir(SCHEMA_ETL == "etl_2021", "Este notebook carrega somente o schema etl_2021.")
    load_dotenv(raiz_projeto / ".env", override=False)
    if os.getenv("DATABASE_URL"):
        url = make_url(os.environ["DATABASE_URL"])
        exigir(url.get_backend_name() == "postgresql", "DATABASE_URL deve apontar para PostgreSQL.")
        url = url.set(drivername="postgresql+psycopg")
    else:
        exigir(bool(os.getenv("PGDATABASE")) and bool(os.getenv("PGUSER")),
               "Configure PGDATABASE e PGUSER no .env ou nas variáveis de ambiente.")
        url = URL.create(
            "postgresql+psycopg", username=os.getenv("PGUSER"),
            password=os.getenv("PGPASSWORD"), host=os.getenv("PGHOST", "localhost"),
            port=int(os.getenv("PGPORT", "5432")), database=os.getenv("PGDATABASE"),
        )
    engine = create_engine(url, echo=False, hide_parameters=True)
    tipos_tratada = {campo: sat.Text() for campo in df_tratada.columns}
    for campo in ["partida_prevista", "partida_real", "chegada_prevista", "chegada_real"]:
        tipos_tratada[campo] = sat.DateTime(timezone=False)
    tipos_tratada["referencia"] = sat.Date()
    for campo in ["linha_origem", "ano_origem", "numero_assentos"]:
        tipos_tratada[campo] = sat.Integer()
    for campo in ["diferenca_partida_min", "diferenca_chegada_min", "atraso_partida_min", "atraso_chegada_min"]:
        tipos_tratada[campo] = sat.Numeric(16, 2)
    for campo in ["partida_atipica_flag", "chegada_atipica_flag"]:
        tipos_tratada[campo] = sat.Boolean()

    def tipos_auditoria_bruta(base):
        tipos = {campo: sat.Text() for campo in base.columns}
        for campo in ["ano_origem", "linha_origem", "linha_preservada"]:
            if campo in tipos:
                tipos[campo] = sat.Integer()
        if "codeshare_presente_no_arquivo" in tipos:
            tipos["codeshare_presente_no_arquivo"] = sat.Boolean()
        if "motivos_rejeicao" in tipos:
            tipos["motivos_rejeicao"] = ARRAY(sat.Text())
        return tipos

    tabelas_carga = {
        "vra_tratada": (df_tratada, tipos_tratada),
        "vra_rejeicoes": (df_rejeicoes, tipos_auditoria_bruta(df_rejeicoes)),
        "vra_duplicatas_excluidas": (df_duplicatas, tipos_auditoria_bruta(df_duplicatas)),
        "vra_anomalias": (vra_anomalias, {
            "arquivo_origem": sat.Text(), "linha_origem": sat.Integer(), "ano_origem": sat.Integer(),
            "campo": sat.Text(), "valor_original": sat.Text(), "motivo": sat.Text(),
        }),
        "vra_conflitos_codigo": (vra_conflitos_codigo, {
            "tipo": sat.Text(), "codigo": sat.Text(), "descricao": sat.Text(), "ocorrencias": sat.BigInteger(),
        }),
        "inventario_arquivos": (inventario, None),
        "resumo_qualidade": (resumo_qualidade, None),
    }
    nomes_tabelas = list(tabelas_carga)
    try:
        with engine.begin() as conexao:
            conexao.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{SCHEMA_ETL}"'))
            existentes = set(conexao.execute(text(
                "SELECT table_name FROM information_schema.tables WHERE table_schema=:schema"
            ), {"schema": SCHEMA_ETL}).scalars())
            exigir(not existentes.intersection(nomes_tabelas) or SUBSTITUIR_CARGA_POSTGRES,
                   "Há tabelas deste pipeline no schema; a substituição está desativada.")
            if SUBSTITUIR_CARGA_POSTGRES:
                for nome in nomes_tabelas:
                    conexao.execute(text(f'DROP TABLE IF EXISTS "{SCHEMA_ETL}"."{nome}"'))
            for nome, (base, tipos) in tabelas_carga.items():
                base.to_sql(nome, con=conexao, schema=SCHEMA_ETL, if_exists="fail",
                            index=False, dtype=tipos, chunksize=2000)
                quantidade = conexao.execute(text(
                    f'SELECT COUNT(*) FROM "{SCHEMA_ETL}"."{nome}"'
                )).scalar_one()
                exigir(quantidade == len(base), f"Contagem incorreta após carregar {nome}.")
            for nome in ["vra_tratada", "vra_rejeicoes", "vra_duplicatas_excluidas"]:
                conexao.execute(text(
                    f'ALTER TABLE "{SCHEMA_ETL}"."{nome}" '
                    'ADD PRIMARY KEY (arquivo_origem, linha_origem)'
                ))
        status_carga_postgresql = {
            "status": "concluida", "schema": SCHEMA_ETL,
            "contagens": {nome: len(base) for nome, (base, _) in tabelas_carga.items()},
        }
        print("Carga e contagens PostgreSQL concluídas no schema", SCHEMA_ETL)
    finally:
        engine.dispose()
else:
    print("Carga PostgreSQL não executada. Revise os resultados antes de habilitá-la.")

if EXPORTAR_EVIDENCIAS:
    (PASTA_SAIDA / "status_carga_postgresql.json").write_text(
        json.dumps(status_carga_postgresql, ensure_ascii=False, indent=2), encoding="utf-8"
    )

Carga PostgreSQL não executada. Revise os resultados antes de habilitá-la.


## 19. O que foi feito e o que decidir depois

**Implementado:** leitura dos 12 arquivos e dos dois layouts; hashes; preservação
de `Codeshare`; deduplicação dos 20 campos brutos com vínculo ao representante;
nomes canônicos; textos e nulos padronizados; dois formatos de referência; datas e
assentos validados; rejeições com todos os motivos; anomalias opcionais; métricas
em minutos; flags de 24 horas; conflitos de descrição; indicadores e conciliação
mensal/anual; exportação das evidências; carga PostgreSQL opcional e transacional.

**Como ler os resultados:** `resumo_qualidade` resume os indicadores, `mensal`
explica a movimentação por arquivo, `df_rejeicoes` conserva os registros rejeitados,
`vra_anomalias` registra problemas opcionais e `vra_conflitos_codigo` apresenta as
variantes que precisam de uma decisão coletiva para as dimensões.

**Próximas decisões:**

1. Revisar os resultados com Murilo e obter o inventário com os hashes e a tabela
   tratada do ELT para a comparação por ocorrência, além dos totais.
2. Configurar e executar a carga no PostgreSQL, caso ainda esteja desativada.
3. Resolver conflitos de descrições com o grupo e integrar dimensões e Fato dos
   três anos, sem somar as saídas ETL e ELT como se fossem voos diferentes.
4. Executar a versão final no Colab e selecionar evidências para relatório e slides.
   O limiar coletivo de voo atrasado e a interpretação de fuso continuam pendentes.

Os resultados são calculados pelos dados de entrada. Nenhum total de 2019 ou do
ELT é usado para filtrar linhas ou forçar a reconciliação.